# 02 - Preprocessing Teks

**Tahap 2** metodologi: membersihkan teks ulasan sebelum dianalisis.

Tahapan yang dijalankan berurutan:

1. Case folding (huruf kecil)
2. Penghapusan URL, tag HTML (`<br>`), mention, hashtag
3. Penghapusan emoji dan angka
4. Penghapusan tanda baca
5. Normalisasi kata tidak baku dan singkatan
6. Tokenizing
7. Stopword removal
8. Stemming dengan Sastrawi

Seluruh logika berada di `src/preprocessing.py`; notebook hanya memanggil dan
menampilkan hasil agar mudah diaudit.

In [1]:
import sys
from pathlib import Path

# Menemukan root proyek agar `import src` berjalan dari mana pun notebook dibuka.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

print("Root proyek:", ROOT)

Root proyek: D:\hikam\tugas DataMining


## 2.1 Memuat data bersih dari Tahap 1

In [2]:
from src import io_utils, preprocessing, normalizer

clean = io_utils.load_csv(config.CLEAN_CSV)
print(f"Ulasan yang akan diproses: {len(clean):,}")
clean[["review_id", "Rating", "Review"]].head(3)

Ulasan yang akan diproses: 6,795


,review_id,Rating,Review
0,1,5,"Ramai banget kalo setiap datang kesini, tapi tetap dicariin kursi kok sama pelayannya...."
1,2,1,"Mbah parkiranya nyebelin banget, udah bayar kok ditagih lagi mana ga bantuin narik mot..."
2,3,3,Pengalaman udh laper..pengen makan bgt...pesen nasgor magelangan...tp rasa nasgornya a...


## 2.2 Kamus normalisasi kata tidak baku

Ulasan Google Maps dipenuhi singkatan dan typo (`gak`, `bgt`, `pelayanannya`,
`enak2`). Normalisasi menyatukan bentuk-bentuk ini agar tidak terpecah menjadi
fitur berbeda saat TF-IDF.

Kamus lengkap: `src/normalizer.py` (`SLANG_MAP`).

In [3]:
slang = pd.DataFrame(normalizer.export_slang_frame())
print(f"Jumlah entri kamus normalisasi: {len(slang):,}")
slang.sample(15, random_state=config.RANDOM_STATE)

Jumlah entri kamus normalisasi: 299


,Kata Asli,Bentuk Baku
281,udh,sudah
265,temen,teman
164,menunya,menu
9,ayam,ayam
77,goreng,goreng
278,trus,terus
93,insyaallah,semoga
109,kayak,seperti
5,anak2,anak
173,murah2,murah


## 2.3 Menjalankan pipeline preprocessing

Proses stemming di-cache ke `data/cache/stem_cache.json`. Pada eksekusi pertama
proses memakan beberapa menit; eksekusi ulang menjadi jauh lebih cepat karena
setiap kata hanya di-stem sekali.

In [4]:
proc, stage_stats, extras = preprocessing.run_preprocessing(use_stemming=True)

[preprocessing] 6795 dokumen diproses
[preprocessing] kosakata akhir: 6517 kata
[preprocessing] cache stemming: 7737 entri
[preprocessing] dokumen kosong setelah stemming: 161


## 2.4 Statistik per tahap

Tabel ini menjadi bukti kuantitatif proses preprocessing di laporan: berapa token
yang hilang pada setiap tahap.

In [5]:
stage_stats

,Tahap,Jumlah Token,Token Unik (Vocab),Jumlah Dokumen,Rata-rata Token/Dokumen,Token Terbuang
0,1. Ulasan mentah (berteks & unik),114989,19306,6795,16.92,0
1,2. Case folding,114989,17384,6795,16.92,0
2,"3. Hapus URL, tag HTML, mention, hashtag",116660,16272,6795,17.17,0
3,4. Hapus emoji & angka,115541,15325,6795,17.00,1119
4,5. Hapus tanda baca,116146,9490,6795,17.09,0
5,6. Normalisasi kata tidak baku,115873,8990,6795,17.05,273
6,7. Tokenizing,115873,8990,6795,17.05,0
7,8. Stopword removal,53206,7737,6795,7.83,62667
8,9. Stemming (Sastrawi),53206,6517,6795,7.83,0


In [6]:
from src import visualize

visualize.plot_preprocessing_stages(stage_stats)

WindowsPath('D:/hikam/tugas DataMining/results/figures/02_tahapan_preprocessing.png')

## 2.5 Contoh hasil tiap tahap

Lima ulasan pertama ditampilkan utuh dari teks mentah sampai teks siap analisis.

In [7]:
examples = extras["examples"]
examples[["Rating", "Mentah", "Setelah Cleaning", "Setelah Normalisasi", "Setelah Stemming"]]

,Rating,Mentah,Setelah Cleaning,Setelah Normalisasi,Setelah Stemming
0,5,"Ramai banget kalo setiap datang kesini, tapi tetap dicariin kursi kok sama pelayannya....",ramai banget kalo setiap datang kesini tapi tetap dicariin kursi kok sama pelayannya m...,ramai banget kalau setiap datang ke sini tapi tetap dicariin kursi kok sama pelayan ma...,ramai dicariin kursi layan makan minum enak harga golong affordable live musik asik no...
1,1,"Mbah parkiranya nyebelin banget, udah bayar kok ditagih lagi mana ga bantuin narik mot...",mbah parkiranya nyebelin banget udah bayar kok ditagih lagi mana ga bantuin narik moto...,mbah parkiranya nyebelin banget sudah bayar kok ditagih lagi mana tidak bantuin narik ...,mbah parkiranya nyebelin bayar tagih bantuin narik motor susah
2,3,Pengalaman udh laper..pengen makan bgt...pesen nasgor magelangan...tp rasa nasgornya a...,pengalaman udh laper pengen makan bgt pesen nasgor magelangan tp rasa nasgornya astagh...,pengalaman sudah laper pengen makan banget pesan nasgor magelangan tapi rasa nasgornya...,alam laper makan pesan nasgor magelangan nasgornya astaghfirullah nasi nasi murah basi...
3,1,"petugas parkirnya bintang 1, sudah bayar ditagih lagi berkali"", mending klo bantuin na...",petugas parkirnya bintang sudah bayar ditagih lagi berkali mending klo bantuin narik k...,petugas parkir bintang sudah bayar ditagih lagi berkali mending kalau bantuin narik ke...,tugas parkir bintang bayar tagih kali mending bantuin narik keluarin kendara narik uan...
4,2,"Gk tauu masaknyaa di tinggal ngapain , gosong gini dii sajikan",gk tauu masaknyaa di tinggal ngapain gosong gini dii sajikan,tidak tauu masaknyaa di tinggal ngapain gosong begini dii sajikan,tauu masaknyaa tinggal ngapain gosong saji


## 2.6 Kosakata akhir

Kata yang paling sering muncul setelah seluruh tahap preprocessing.

In [8]:
vocab = extras["vocab_counts"]
vocab.head(30).to_frame("Frekuensi")

,Frekuensi
makan,1592
musik,1540
nongkrong,1499
live,1339
enak,1305
harga,938
nyaman,872
layan,849
kopi,782
menu,710


In [9]:
visualize.plot_top_words(vocab, n=30, title="30 Kata Paling Sering Muncul Setelah Preprocessing")

WindowsPath('D:/hikam/tugas DataMining/results/figures/02_kata_terbanyak.png')

In [10]:
visualize.plot_wordcloud(vocab)

WindowsPath('D:/hikam/tugas DataMining/results/figures/02_wordcloud.png')

## 2.7 Pemeriksaan kualitas

Beberapa pemeriksaan untuk memastikan preprocessing tidak merusak data:

In [11]:
cek = pd.DataFrame(
    {
        "Pemeriksaan": [
            "Dokumen kosong setelah stopword",
            "Dokumen kosong setelah stemming",
            "Rata-rata token sebelum stopword",
            "Rata-rata token setelah stemming",
            "Token unik akhir (kosakata)",
            "Entri cache stemming",
        ],
        "Nilai": [
            int((proc["tokens"].str.split().str.len() == 0).sum()),
            int((proc["n_tokens_after"] == 0).sum()),
            round(float(proc["n_tokens_before"].mean()), 2),
            round(float(proc["n_tokens_after"].mean()), 2),
            len(vocab),
            extras["cache_size"],
        ],
    }
)
io_utils.save_csv(cek, config.TABLES_DIR / "02_pemeriksaan_kualitas.csv")
cek

,Pemeriksaan,Nilai
0,Dokumen kosong setelah stopword,161.00
1,Dokumen kosong setelah stemming,161.00
2,Rata-rata token sebelum stopword,7.83
3,Rata-rata token setelah stemming,7.83
4,Token unik akhir (kosakata),6517.00
5,Entri cache stemming,7737.00


## 2.9 Catatan implementasi

- Cache stemming (`data/cache/stem_cache.json`) menyimpan pemetaan
  kata → kata dasar sehingga eksekusi ulang notebook tidak mengulang proses
  yang paling mahal.
- Kolom `text_clause` menyimpan teks yang sudah dinormalisasi **tetapi masih
  mempertahankan tanda baca**. Kolom ini dipakai pada analisis sentimen per
  aspek, karena tanda baca dan kata seperti *tapi* / *namun* adalah penanda
  batas antar-aspek.
- Kolom `tokens_stemmed` dipakai untuk TF-IDF dan ekstraksi kata kunci aspek.

## 2.10 Kesimpulan Tahap 2

- Teks ulasan telah melalui 8 tahap preprocessing dan tersimpan pada kolom
  `tokens_stemmed`.
- Dokumen tanpa token setelah stemming dibiarkan di dataset (tidak dibuang) agar
  jumlah ulasan tetap konsisten; dokumen ini tidak menyumbang fitur TF-IDF.
- Artefak yang dihasilkan:
  - `data/processed/02_reviews_preprocessed.csv`
  - `results/tables/02_preprocessing_stages.csv`
  - `results/tables/02_preprocessing_examples.csv`
  - `data/cache/stem_cache.json`